# Resposta del consum elèctric de Barcelona a la temperatura
### Notebook v0.14 · article 1 · resposta a la revisió externa del 24/10/2026

**Tesi:** *la ciutat observada a través de les seves respostes* (Pau Buchaca, ETSAB-UPC). Peça P1. **Tot és a escala de ciutat** (suma dels codis postals), no per codi postal.

**Què afirma l'article (i on es comprova en aquest notebook)**

| # | Resultat | Secció |
|---|---|---|
| R1 | La ciutat respon a la calor amb ~6 % de consum residencial per grau-dia, amb memòria d'1-2 dies; els dies d'onada, fora de l'ajust, queden ben predits | 7 |
| R2 | La resposta del **2024 és més gran que la del 2021** (una sola mètrica, amb interval per *bootstrap* que inclou la tria de $T_c$). Amb quatre estius no es pot distingir una tendència d'un salt | 8 |
| R3 | Les corbes de cada estiu són **compatibles amb** la mateixa forma multiplicada per un factor (model de llindars) | 9 |
| R4 | **Indici** (decisió de Pau, 25/10): a la franja de 0 a 6 h, la resposta a la calor de nit creix més que la de dia. Depèn de l'estiu 2024 | 10 |
| R5 | El fred: nivell de la resposta elèctrica al fred (~3,7 %/K). Ni el canvi de $\chi_h$ ni la pèrdua de simetria de la V són robustos | 11 |
| R6 | **No hi ha histèresi** estreta | 12 |
| — | Sensibilitat del **component elèctric** de la calor antropogènica, $\partial Q_F/\partial T$ | 13 |
| — | **Predicció registrada** de l'estiu 2026 (validació fora de mostra) | 14 |

**Model**
$$\ln E(d) = \sum_{k=0}^{3} \chi(k)\,\mathrm{CDD}(d-k) + \text{any} + \text{setmana ISO} + \text{dia de la setmana} + \text{festiu} + \varepsilon,\qquad \mathrm{CDD}=\max(T-T_c,0)$$
$\chi^0=\sum_k\chi(k)$ és el % de consum de més per cada grau-dia. Errors per clústers de setmana ISO.

**Què canvia respecte de la v0.13 (revisió del 24/10):**
- **Regla de qualitat per any** (secció 5), escrita i aplicada igual a tots els estius. El 2025 no la passa i surt de totes les estimacions principals; es mostra a part, amb un altre estil.
- **Una sola mètrica de creixement**: el quocient $\chi^0_{2024}/\chi^0_{2021}$, amb interval de *bootstrap* per blocs de setmana que refà la tria de $T_c$. λ queda com a comprovació de forma.
- **R4:** test de la diferència nit − dia amb la covariància completa, control de radiació (autoconsum fotovoltaic), dies laborables i caps de setmana (tarifa 2.0TD), graella de llindars i nombre de dies del règim A.
- **Temperatura de les estacions del Meteocat** com a prova de robustesa.
- **Predicció del 2026** registrada abans de tenir-ne les dades.

**Com usar-lo:** `Entorn d'execució → Executa-ho tot`. Totes les baixades tenen memòria cau a `dades_tesi`. El notebook **s'atura** si una alerta coneguda no queda tractada. Al final desa `dades_per_claude_v14.zip` al Drive.

## 1. Instal·lació

Instal·la el paquet de la tesi directament de GitHub. Si el repositori canvia, tornar a executar aquesta cel·la porta la versió nova.

In [ ]:
!pip install -q holidays "git+https://github.com/PauBuchacaMartin/power_temperature_urban@main"

import os, json, zipfile, shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import holidays
from scipy import stats
import consum_temperatura as ct
from consum_temperatura import figstyle as fs

fs.apply_style()
pd.set_option("display.width", 160)
print(f"[OK] consum_temperatura {ct.__version__}")

## 2. Especificació congelada

**Què fa:** fixa totes les decisions de l'anàlisi. Cada decisió presa **després** de veure resultats porta la marca *post hoc* i la data.

**Per què:** és la regla que protegeix l'article de triar l'especificació que dona el resultat desitjat (el «jardí de camins que es bifurquen»).

In [ ]:
SPEC = {
    # Dades
    "carpeta": "/content/drive/MyDrive/dades_tesi",
    "anys": [2021, 2022, 2023, 2024, 2025],     # anys descarregats; quins entren ho decideix la regla de qualitat
    "lat_lon": (41.39, 2.17),                  # ERA5 (Open-Meteo), una sola sèrie per a la ciutat
    "centre_estacions": (41.401, 2.1644),      # mitjana dels centres dels CP: on s'interpolen les estacions
    # Regla de qualitat per any (secció 5). POST HOC (24/10/2026): escrita després de veure la χ de
    # la v0.13, però només fa servir mètriques de dades (no la resposta) i s'aplica igual a tots els anys.
    # Queda congelada per al 2026 i següents.
    "regla_min_cdd": 0.90,                     # un estiu entra si >= 90 % dels seus graus-dia cauen en dies útils
    # Model principal (R1, R2)
    "mesos_estiu": [6, 7, 8, 9],
    "retards": 3,                              # memòria: dies 0..3
    "tc_candidats": list(np.arange(18.0, 25.5, 0.5)),
    "onada_percentil": 0.95, "onada_dies": 3,  # onada: Tmax > P95 de l'estiu durant >= 3 dies
    "any_ref": 2021, "any_fi": 2024,
    "bootstrap_n": 400, "bootstrap_llavor": 1, # blocs = setmanes ISO, dins de cada estiu
    # Dia–nit (R4)
    "tc_dia": 27.0, "tc_nit": 20.0,            # T 12-18 h i T 0-6 h (fixats a la v0.10)
    "graella_tc_dia": [25.0, 26.0, 27.0, 28.0, 29.0],
    "graella_tc_nit": [18.0, 19.0, 20.0, 21.0, 22.0],
    # Fred (R5)
    "hiverns": [2021, 2022, 2023, 2024],       # hivern 2021 = nov. 2021 - març 2022
    "mesos_hivern": [11, 12, 1, 2, 3],         # especificació inicial (v0.12): la del cos de l'article
    "mesos_hivern_cor": [12, 1, 2],            # POST HOC (24/10/2026): només a l'annex
    "th_candidats": list(np.arange(10.0, 19.5, 0.5)),
    # Qualitat de dades
    "anomalia_k": 6.0, "bloc_llindar": 0.12, "bloc_dies": 7,
    # Calor antropogènica
    "area_km2": 101.0,                         # àrea dels 42 CP (reconstrucció per Voronoi d'adreces, v0.2.3)
    "h_eff": (10.0, 50.0),                     # W m-2 K-1 · NO VERIFICAT: només per a un ordre de magnitud de G
}
ANYS, LAGS = SPEC["anys"], SPEC["retards"]
A0, A1 = SPEC["any_ref"], SPEC["any_fi"]
os.makedirs("figures_v14", exist_ok=True)
print(json.dumps({k: v for k, v in SPEC.items() if k not in ("tc_candidats", "th_candidats")}, indent=1, ensure_ascii=False))

## 3. Dades

**Què fa:** llegeix els CSV de consum elèctric d'Open Data BCN (origen Datadis) de la carpeta `dades_tesi`, la temperatura ERA5 diària i horària, la humitat i el preu (de la memòria cau), i els festius de Catalunya.

**Què cal mirar:** que el consum anual del residencial sigui d'uns 2.000 GWh. El portal diu que els valors són MWh, però són **kWh**: amb MWh, el residencial de Barcelona seria ~2.000 TWh/any, gairebé deu vegades tot el consum d'Espanya. La cel·la ho comprova sola.

In [ ]:
from pathlib import Path
if SPEC["carpeta"].startswith("/content/drive") and not Path("/content/drive/MyDrive").exists():
    from google.colab import drive
    drive.mount("/content/drive")
CARPETA = SPEC["carpeta"]

# --- Consum: un CSV per any, tal com el dona el portal ---
trossos = []
for f in sorted(Path(CARPETA).glob("*.csv")):
    if "consum_electricitat" in f.name.lower() and any(str(a) in f.name for a in ANYS):
        print("  llegint", f.name)
        trossos.append(ct.daily_consumption(pd.read_csv(f)))
assert trossos, f"No hi ha CSV de consum a {CARPETA} (han de contenir 'consum_electricitat' i l'any al nom)"
df = pd.concat(trossos, ignore_index=True)
df = df[df["data"].dt.year.isin(ANYS)]
assert len(df), "Els CSV s'han llegit però no en surt cap fila: revisa els noms de les columnes"
print(f"[OK] Consum: {len(df):,} files · {df['data'].min().date()} → {df['data'].max().date()} · "
      f"{df['cp'].nunique()} CP · sectors {sorted(df['sector'].unique())}")

# Comprovació d'unitats (alerta 6): residencial anual entre 1.000 i 4.000 GWh si els valors són kWh
gwh = df[df["sector"] == "Residencial"].groupby(df["data"].dt.year)["kwh"].sum() / 1e6
print("Residencial anual (GWh, si els valors són kWh):", gwh.round(0).to_dict())
assert gwh.loc[[a for a in gwh.index if a <= 2024]].between(1000, 4000).all(), "Unitats inesperades: revisa el CSV"

# --- Temperatura diària i horària (ERA5, Open-Meteo), amb memòria cau ---
lat, lon = SPEC["lat_lon"]
fi = df["data"].max().strftime("%Y-%m-%d")
temp = ct.from_cache(CARPETA, "temperatura.csv", lambda: ct.era5_daily(lat, lon, f"{ANYS[0]}-01-01", fi), until=fi)
temp = temp[["data", "tmax", "tmean"]].dropna().sort_values("data").reset_index(drop=True)
hor = ct.from_cache(CARPETA, "temp_horaria.csv", lambda: ct.era5_hourly(lat, lon, f"{ANYS[0]}-01-01", fi), date_col="hora")
hor = hor.rename(columns={"temperature_2m": "temp"})[["hora", "temp"]].dropna()

# --- Humitat i preu: només de la memòria cau (els va baixar la v0.11); si no hi són, les proves se salten ---
def opcional(nom):
    ruta = f"{CARPETA}/{nom}"
    return pd.read_csv(ruta, parse_dates=["data"]) if os.path.exists(ruta) else None
humitat, preus = opcional("humitat_era5.csv"), opcional("preus_electricitat.csv")
print("Humitat:", "sí" if humitat is not None else "NO", "· Preu:", "sí" if preus is not None else "NO")

FESTIUS = sorted(pd.to_datetime(list(holidays.country_holidays("ES", subdiv="CT", years=ANYS).keys())))
temp["onada"] = ct.heatwaves(temp, SPEC["onada_percentil"], SPEC["onada_dies"]).values
ONADES = set(temp.loc[temp["onada"], "data"])
print(f"[OK] Temperatura: {len(temp)} dies · dies d'onada: {len(ONADES)}")

# --- v0.14: radiació solar diària (prova de l'autoconsum fotovoltaic, secció 10) ---
try:
    rad = ct.from_cache(CARPETA, "radiacio.csv", lambda: ct.era5_radiation(lat, lon, f"{ANYS[0]}-01-01", fi), until=fi)
except Exception as e:
    rad = None; print(f"[!] Sense radiació ({e}): la prova FV se salta")

# --- v0.14: estacions del Meteocat (XEMA), temperatura independent d'ERA5 (secció 8) ---
def baixa_xema():
    import requests, io
    llista = ",".join(f"'{c}'" for c in ct.XEMA_CITY)
    params = {"$select": "codi_estacio,data_lectura,codi_variable,valor",
              "$where": (f"codi_estacio in({llista}) AND codi_variable in('1000','1001') "
                         f"AND data_lectura >= '{ANYS[0]}-01-01T00:00:00' AND data_lectura <= '{fi}T00:00:00'"),
              "$limit": 500000}
    r = requests.get(ct.XEMA_URL, params=params, timeout=300); r.raise_for_status()
    return pd.read_csv(io.StringIO(r.text))
try:
    xema = ct.from_cache(CARPETA, "xema.csv", baixa_xema, date_col="data_lectura", until=fi)
    T_EST, pesos_est = ct.station_temperature(ct.xema_daily(xema), *SPEC["centre_estacions"], fallback=temp)
    print("[OK] Estacions: pesos", dict(zip(ct.XEMA_CITY, pesos_est.round(2))))
except Exception as e:
    T_EST = None; print(f"[!] Sense estacions ({e}): la prova se salta")

## 4. Qualitat de les dades i registre d'alertes

**Què fa, en quatre passos:**
1. **Dies complets:** un dia entra només si hi són tots els CP actius aquell trimestre (si no, la suma de la ciutat surt baixa).
2. **Auditoria per sectors:** dibuixa el consum diari de tota la ciutat per sector. Aquí es veuen les alertes 3 i 4.
3. **Anomalies condicionades a la temperatura**, per a cada sèrie (residencial, serveis, total). Un model flexible (graus d'1 °C + any + setmana + dia + festiu) explica la V, i el que queda fora és sospitós:
   - **dies sols** amb un residu > 6 σ, excepte els dies de calor o fred extrems, que mai s'exclouen (són el senyal);
   - **blocs negatius**: una mediana mòbil de 15 dies per sota de −12 % durant ≥ 7 dies, que vol dir que falten subministraments.

   Els blocs positius es mostren però no s'exclouen: dues setmanes de calor persistent (agost del 2024) apugen el consum per sobre del model diari.
4. **Règim de les franges horàries** (alerta 2).

**Comprovacions automàtiques:** el notebook s'atura si el forat dels serveis del 2023 no queda exclòs o si s'exclou algun dia d'onada del residencial.

**Què cal mirar:** la taula-registre del final. Cada alerta hi té el seu efecte i la seva solució.

In [ ]:
# 4.1 Dies complets (amb els CP del residencial)
res_cp = df[df["sector"] == "Residencial"].groupby(["data", "cp"], as_index=False)["kwh"].sum()
COMPLETS = set(ct.complete_days(res_cp))
print(f"Dies complets: {len(COMPLETS)} de {res_cp['data'].nunique()}")

# 4.2 Auditoria per sectors (alertes 3 i 4)
sect = df.groupby(["data", "sector"])["kwh"].sum().unstack().fillna(0) / 1e3
fig, axs = plt.subplots(len(sect.columns) + 1, 1, figsize=(12, 2.1 * (len(sect.columns) + 1)), sharex=True)
for ax, col in zip(axs, list(sect.columns) + ["Total"]):
    y = sect.sum(axis=1) if col == "Total" else sect[col]
    ax.plot(y.index, y, lw=0.5, color=fs.CAT[0]); ax.plot(y.index, y.rolling(28, center=True).median(), lw=1.2, color=fs.INK)
    ax.set_ylabel(f"{col}\nMWh/dia", fontsize=8)
axs[0].set_title("Consum diari de tota la ciutat per sector (línia negra: mediana de 28 dies)")
plt.tight_layout(); plt.savefig("figures_v14/auditoria_sectors.png"); plt.show()
print("Quota de cada sector (%) per trimestre:")
print(ct.sector_shares(df).round(1).to_string())

In [ ]:
# 4.3 Anomalies condicionades a la temperatura, per a cada sèrie
EXTREMS = ct.extreme_days(temp)
kw_an = dict(k=SPEC["anomalia_k"], block_threshold=SPEC["bloc_llindar"], block_min_days=SPEC["bloc_dies"], protect=EXTREMS)
SERIES, DIES_BONS, ANOMALIES = {}, {}, {}
for nom, sec in [("Residencial", "Residencial"), ("Serveis", "Serveis"), ("Total", None)]:
    s = ct.city_series(df, sector=sec)
    bons, an = ct.clean_days(s, temp, FESTIUS, complete=COMPLETS, **kw_an)
    SERIES[nom], DIES_BONS[nom], ANOMALIES[nom] = s, bons, an
    print(f"\n{nom}: {len(bons)} dies bons · {int(an['single'].sum())} dies sols i {int(an['block'].sum())} dies de bloc exclosos")
    blocs = ct.anomaly_blocks(an)
    if len(blocs): print("  Blocs negatius (exclosos):\n" + blocs.to_string(index=False))
    pos = ct.anomaly_blocks(an, "block_pos")
    if len(pos): print("  Blocs positius (NO exclosos):\n" + pos.to_string(index=False))
    if an["protected"].any(): print("  Dies extrems marcats però no exclosos:", [str(d.date()) for d in an.index[an["protected"]]])

# Sèries netes per a l'anàlisi (les mateixes en totes les seccions)
NET = {nom: s[s["data"].isin(DIES_BONS[nom])].reset_index(drop=True) for nom, s in SERIES.items()}

# Comprovacions automàtiques: les alertes conegudes han de quedar tractades
forat = pd.date_range("2023-01-15", "2023-03-15")
assert not NET["Serveis"]["data"].isin(forat).any(), "ALERTA 3 no tractada: el forat dels serveis del 2023 és a l'anàlisi"
assert not (set(SERIES["Residencial"]["data"]) - DIES_BONS["Residencial"]) & ONADES & COMPLETS, "S'ha exclòs un dia d'onada del residencial"
print("\n[OK] Alerta 3 tractada (forat dels serveis exclòs) i cap dia d'onada del residencial exclòs")
pd.concat({n: a[a["single"] | a["block"]] for n, a in ANOMALIES.items()}).to_csv("dies_exclosos_v14.csv")

In [ ]:
# 4.4 Règim de les franges horàries (alerta 2)
FRANGES = sorted(t for t in df["tram"].dropna().unique() if "consta" not in str(t).lower())
NOM_F = {t: f"{int(str(t).split(':')[0][-2:]):02d}-{int(str(t).split(':')[0][-2:]) + 6:02d}" for t in FRANGES}
pb = (df[(df["sector"] == "Residencial") & df["tram"].isin(FRANGES)]
      .pivot_table(index="data", columns="tram", values="kwh", aggfunc="sum").rename(columns=NOM_F))
anom_f, REGIM_B = ct.band_regime(pb, evening="18-24", morning="06-12")
print("Dies del règim B (repartiment entre franges canviat) per any:",
      pd.Series([d.year for d in REGIM_B]).value_counts().sort_index().to_dict())

# 4.5 Registre d'alertes
REGISTRE = pd.DataFrame([
    ["1", "Recodificació de CP (tardor 2023, juliol 2025)", "per CP; no a la ciutat", "tot a escala de ciutat; creixement 2021→2024; el 2025 només com a robustesa"],
    ["2", "Canvi del repartiment entre franges (2024-25)", f"{len(REGIM_B)} dies", "R4 només amb dies del règim A (principal) i amb variable de règim (secundari); sense 2025"],
    ["3", "Forat de cobertura dels serveis (gen.-mar. 2023)", f"{int(ANOMALIES['Serveis']['block'].sum())} dies de serveis", "exclòs pel detector de blocs; també el del residencial de principis de gener"],
    ["4", "Reclassificacions de sector (ago. 2023; set. 2025)", "serveis i indústria", "prova de composició amb el total de tots els sectors"],
    ["5", "Dies dolents i bloc d'agost del 2025", f"{int(ANOMALIES['Residencial']['single'].sum() + ANOMALIES['Residencial']['block'].sum())} dies del residencial", "detector condicionat a la temperatura; els dies extrems mai s'exclouen"],
    ["6", "Unitats: el portal diu MWh, són kWh", "—", "comprovació de l'ordre de magnitud (secció 3)"],
    ["7", "Sense validació externa del total", "—", "limitació declarada; demanar el total anual a l'Ajuntament o l'ICAEN"],
], columns=["alerta", "què", "abast", "tractament"])
print(REGISTRE.to_string(index=False))
REGISTRE.to_csv("registre_alertes_v14.csv", index=False)

## 5. Regla de qualitat per any

**Què fa:** per a cada estiu, mesura quina part dels **graus-dia de refrigeració** (CDD amb $T_c$ = 20 °C) cau en dies útils, és a dir, complets i no exclosos pel detector d'anomalies. Ho fa per a la sèrie diària i per a les franges horàries (dies útils i del règim A).

**La regla:** un estiu entra a les estimacions principals si **≥ 90 % dels seus CDD** cauen en dies útils. Es fan servir els CDD i no els dies perquè la resposta s'identifica amb els dies calents: perdre deu dies frescos costa poc i perdre deu dies calents costa molt.

**Honestedat sobre la regla:** està escrita el 24/10/2026, **després** de veure que el 2025 no seguia la tendència. Per això:
- només fa servir mètriques de dades, no la resposta;
- s'aplica igual a tots els anys;
- queda congelada per al 2026 i següents (secció 14);
- tots els resultats es donen també **amb** el 2025, com a robustesa.

El $T_c$ = 20 °C de la regla és el de la v0.13. Si el 2025 entrés a la tria de $T_c$, la regla dependria de si hi entra.

**Què cal mirar:** si la separació és clara (lluny del 90 %) o està al límit.

In [ ]:
COB = ct.summer_coverage(DIES_BONS["Residencial"], temp, 20.0, ANYS)
COB_F = ct.summer_coverage(DIES_BONS["Residencial"] - set(REGIM_B), temp, 20.0, ANYS)
REGLA = pd.concat({"diària": COB, "franges, règim A": COB_F}, axis=1)
print("Cobertura de cada estiu (fracció de dies i de CDD en dies útils):"); print(REGLA.round(3).to_string())

ANYS_P = ct.quality_rule(COB, SPEC["regla_min_cdd"])           # anys de les estimacions principals
ANYS_F = ct.quality_rule(COB_F, SPEC["regla_min_cdd"])         # anys que passen la regla a les franges
FORA = [a for a in ANYS if a not in ANYS_P]
print(f"\n[REGLA] Sèrie diària: entren {ANYS_P} · fora {FORA}")
print(f"[REGLA] Franges (règim A): passen {ANYS_F}")
assert A0 in ANYS_P and A1 in ANYS_P, "L'any de referència o el final no passen la regla: cal repensar R2"
if A1 not in ANYS_F:
    print(f"[!] El {A1} NO passa la regla a les franges: R4 ({A0}→{A1}) es dona amb aquest avís, i també {A0}→{max(ANYS_F)}")
REGLA.to_csv("regla_qualitat_v14.csv")

## 6. Llindar de confort $T_c$

**Què fa:** prova llindars de 18 a 25 °C i tria el que minimitza l'error del model (residencial net, estius que passen la regla, sense els dies d'onada).

**Per què l'error quadràtic mitjà:** el model té un colze a $T_c$. El llindar que s'equivoca menys és l'estimació de mínims quadrats d'on és el colze.

**Què cal mirar:** el mínim és poc marcat entre 19 i 21 °C. Per això la secció 8 posa la tria de $T_c$ **dins** del *bootstrap*.

In [ ]:
RES = NET["Residencial"]
TC, perfil = ct.choose_tc(RES, temp, SPEC["tc_candidats"], LAGS, FESTIUS, exclude=ONADES, years=ANYS_P)
fig, ax = plt.subplots(figsize=(6, 3.2))
ax.plot(perfil["tc"], perfil["mse"], "o-"); ax.axvline(TC, ls="--", color=fs.INK_2)
ax.set_xlabel("$T_c$ (°C)"); ax.set_ylabel("Mean squared error"); ax.set_title(f"Comfort threshold: $T_c$ = {TC} °C")
plt.tight_layout(); plt.savefig("figures_v14/tria_tc.png"); plt.show()
print(f"[OK] Tc = {TC} °C · l'error varia un {100 * (perfil['mse'].max() / perfil['mse'].min() - 1):.0f} % entre llindars")

## 7. R1 · La resposta de la ciutat i els dies d'onada

**Què fa, en tres parts:**
1. **Funció de resposta** $\chi(k)$, amb tots els estius que passen la regla junts. **Els dies d'onada són fora de l'ajust.**
2. **Comprovació numèrica per correlacions:** la mateixa $\chi$ calculada amb les covariàncies desfasades entre consum i temperatura. Amb les mateixes variables, regressió i correlacions estimen gairebé el mateix, de manera que **no és una prova independent de linealitat**; només comprova que el càlcul és coherent.
3. **Test de les onades:** el model, amb la $\chi$ de cada estiu i ajustat **sense** els dies d'onada, prediu aquests dies. És fora de mostra en dies, no en anys.

**Què NO s'ha de llegir:** els retards de CDD estan molt correlacionats, de manera que els $\chi(k)$ individuals no són estables. Només $\chi^0$ i la fracció del mateix dia són robustos. No es dona cap lectura física de la forma (inèrcia dels edificis) fins a tenir el nucli exponencial.

**Què cal mirar:** el residu d'onada ha de ser ≈ 0 dins de la dispersió (sense saturació ni amplificació).

In [ ]:
r1 = ct.fit_city(RES, temp, TC, LAGS, FESTIUS, exclude=ONADES, years=ANYS_P)
chi_k = np.array([100 * r1["beta"][f"cdd{k}"] for k in range(LAGS + 1)])
se_k = np.array([100 * np.sqrt(r1["cov"].loc[f"cdd{k}", f"cdd{k}"]) for k in range(LAGS + 1)])
reg, cor = ct.correlation_check(RES, temp, TC, LAGS, ANYS_P, FESTIUS, exclude=ONADES)
ones, ones_any, sd_ones = ct.heatwave_residuals(RES, temp, TC, ONADES, LAGS, ANYS_P, FESTIUS)

print(f"χ⁰ (estius {ANYS_P[0]}-{ANYS_P[-1]}) = {r1['chi0']:.2f} ± {r1['se']:.2f} %/K · {r1['n']} dies (sense onades)")
print("χ(k) %/K (no interpretar un a un):", np.round(chi_k, 2), " · fracció el mateix dia:", f"{chi_k[0] / chi_k.sum():.0%}")
print(f"Comprovació numèrica: χ⁰ regressió {reg.sum():.2f} vs correlacions {cor.sum():.2f} %/K")
print(f"\nDies d'onada (fora de l'ajust), residu mitjà {ones['residu_pct'].mean():+.1f} % "
      f"en {len(ones)} dies (dispersió dels dies normals ±{sd_ones:.1f} %)")
print(ones_any.round(1).to_string())

fig, axs = plt.subplots(1, 2, figsize=(11, 3.6))
axs[0].errorbar(range(LAGS + 1), chi_k, yerr=2 * se_k, fmt="o-", color=fs.CAT[0])
axs[0].axhline(0, color=fs.INK_2, lw=0.8); axs[0].set_xticks(range(LAGS + 1))
axs[0].set_xlabel("Lag k (days)"); axs[0].set_ylabel("χ(k) (% per K)")
axs[0].set_title("a) Response function of the city")
axs[1].scatter(ones["tmean"], ones["residu_pct"], s=18, color=fs.CAT[1])
axs[1].axhline(0, color=fs.INK_2, lw=0.8); axs[1].axhspan(-2 * sd_ones, 2 * sd_ones, color=fs.GRID, zorder=0)
axs[1].set_xlabel("Daily mean temperature (°C)"); axs[1].set_ylabel("Observed − predicted (%)")
axs[1].set_title("b) Heatwave days (not used in the fit)")
plt.tight_layout(); plt.savefig("figures_v14/fig1_response_function.png"); plt.show()

## 8. R2 · Quant més gran és la resposta del 2024 que la del 2021

**La mètrica principal (una sola):** el quocient
$$Q = \chi^0_{2024}/\chi^0_{2021},$$
amb un **interval del 95 % per *bootstrap* per blocs**. Es remostregen setmanes ISO senceres dins de cada estiu (la correlació entre dies consecutius es conserva) i, a cada rèplica, **es torna a triar $T_c$**. Així l'interval inclou la incertesa del llindar, que era la principal font d'incertesa oculta de la v0.13.

**Per què $Q$ i no λ (proposta del revisor):** λ no depèn de $T_c$, però cada corba de la secció 9 s'ancora en els dies ≤ 20 °C del seu estiu, que són només 9-14 per estiu, repartits en 3-4 setmanes. Al *bootstrap*, λ surt molt més inestable que $Q$. Aquí es donen tots dos, i el text de l'article fa servir $Q$ (**decisió de Pau, 25/10/2026**).

**Redacció:** «la resposta del 2024 és més gran que la del 2021», **no** «creix a un ritme». Amb quatre estius no es distingeix una tendència d'un salt. La z de cada any mesura la precisió de l'any, no la variabilitat entre anys.

**Proves de robustesa** (canvia una cosa cada vegada; ara inclouen el 2022→2024, el pendent, la temperatura de les estacions i el 2025):

| Prova | Què descarta |
|---|---|
| Tots els sectors sumats | Reclassificacions de sector i canvis de cobertura (alertes 3 i 4) |
| $T_c$ de 18 a 23 °C | Que depengui del llindar (el *bootstrap* ho resumeix) |
| Temperatura de les estacions (Meteocat) | Que sigui un artefacte d'ERA5 |
| Amb els dies d'onada | La selecció de dies |
| Terme quadràtic comú | Una resposta corbada |
| Punt de rosada / preu PVPC | La humitat / el preu diari |
| Amb el 2025 | La regla de qualitat |
| Només juny i setembre / només juliol i agost | **On** creix (mecanisme) |

**Què cal mirar:** que l'interval de $Q$ no inclogui 1, i que les proves mantinguin el signe. La magnitud varia i per això es dona l'interval.

In [ ]:
def creixement(serie, tc=None, exclude=ONADES, years=None, extra=None, months=None, t=None):
    kw = {} if months is None else {"months": months}
    tab, b, c = ct.response_by_year(serie, temp if t is None else t, TC if tc is None else tc, LAGS,
                                    ANYS_P if years is None else years, FESTIUS, exclude=exclude, extra=extra, **kw)
    d, se = ct.growth(b, c, A0, A1)
    return tab, b, c, d, se

tab_res, b_res, c_res, d0, se0 = creixement(RES)
Q0 = tab_res.loc[A1, "chi0"] / tab_res.loc[A0, "chi0"]
d22, se22 = ct.growth(b_res, c_res, 2022, A1)
pend, se_pend = ct.slope_over_years(b_res, c_res, ANYS_P)
print("Residencial · χ⁰ per estiu (%/K):", {a: f"{r.chi0:.2f} ± {r.se:.2f}" for a, r in tab_res.iterrows()})

# --- 8.1 Bootstrap per blocs de setmana, amb la tria de Tc a dins ---
estius = RES[RES["data"].dt.month.isin(SPEC["mesos_estiu"]) & RES["data"].dt.year.isin(ANYS_P)].reset_index(drop=True)
def estadistic(s):
    tc_b, _ = ct.choose_tc(s, temp, SPEC["tc_candidats"], LAGS, FESTIUS, exclude=ONADES, years=ANYS_P)
    tb, _, _ = ct.response_by_year(s, temp, tc_b, LAGS, ANYS_P, FESTIUS, exclude=ONADES)
    pool = ct.fit_city(s, temp, tc_b, LAGS, FESTIUS, exclude=ONADES, years=ANYS_P)["chi0"]
    try:                                    # λ: si un estiu es queda sense dies ≤ 20 °C, la corba no està definida
        y_b, s_b = ct.response_curves(s, temp, ANYS_P, FESTIUS)
        lam_b = ct.threshold_factor(y_b, s_b, ref=A0)[0][A1]
    except ValueError:
        lam_b = np.nan
    return {"tc": tc_b, "chi0_tots": pool, f"chi0_{A0}": tb.loc[A0, "chi0"], f"chi0_{A1}": tb.loc[A1, "chi0"],
            "Q": tb.loc[A1, "chi0"] / tb.loc[A0, "chi0"], "dif": tb.loc[A1, "chi0"] - tb.loc[A0, "chi0"],
            f"lambda_{A1}": lam_b}
BOOT = ct.block_bootstrap(estius, estadistic, n=SPEC["bootstrap_n"], seed=SPEC["bootstrap_llavor"])
IC = {k: ct.percentile_interval(BOOT[k]) for k in BOOT.columns}
print(f"\nBootstrap: {len(BOOT)} rèpliques ({BOOT.attrs['failed']} fallides) · λ no definida a {int(BOOT[f'lambda_{A1}'].isna().sum())} "
      "(algun estiu sense dies ≤ 20 °C: és la fragilitat de l'ancoratge de les corbes)")
print("Tc triat a les rèpliques:", BOOT["tc"].value_counts().sort_index().to_dict())
print(f"\n>>> MÈTRICA PRINCIPAL: Q = χ⁰_{A1}/χ⁰_{A0} = {Q0:.2f}  [IC 95 %: {IC['Q'][0]:.2f} – {IC['Q'][1]:.2f}]")
print(f"    χ⁰ {A0} = {tab_res.loc[A0, 'chi0']:.2f} [{IC[f'chi0_{A0}'][0]:.2f} – {IC[f'chi0_{A0}'][1]:.2f}] · "
      f"χ⁰ {A1} = {tab_res.loc[A1, 'chi0']:.2f} [{IC[f'chi0_{A1}'][0]:.2f} – {IC[f'chi0_{A1}'][1]:.2f}] %/K")
print(f"    diferència {d0:+.2f} %/K [{IC['dif'][0]:+.2f} – {IC['dif'][1]:+.2f}] (l'error analític ±{se0:.2f} no inclou Tc)")
print(f"    χ⁰ tots els estius = {r1['chi0']:.2f} [{IC['chi0_tots'][0]:.2f} – {IC['chi0_tots'][1]:.2f}] %/K")

# --- 8.2 Proves de robustesa ---
PROVES = {"Base (residencial)": (d0, se0)}
PROVES[f"2022→{A1}"] = (d22, se22)
PROVES["Pendent (per any, ×3)"] = (3 * pend, 3 * se_pend)        # ×3 per comparar-lo amb un canvi de 3 anys
PROVES["Tots els sectors sumats"] = creixement(NET["Total"])[3:]
for tc_ in [18.0, 19.0, 21.0, 22.0, 23.0]:
    PROVES[f"Tc = {tc_:.0f} °C"] = creixement(RES, tc=tc_)[3:]
if T_EST is not None:
    TC_EST, _ = ct.choose_tc(RES, T_EST, SPEC["tc_candidats"], LAGS, FESTIUS, exclude=ONADES, years=ANYS_P)
    PROVES[f"Estacions Meteocat (Tc = {TC_EST:.1f} °C, triat)"] = creixement(RES, tc=TC_EST, t=T_EST)[3:]
    PROVES[f"Estacions Meteocat (Tc = {TC:.0f} °C)"] = creixement(RES, t=T_EST)[3:]
PROVES["Amb els dies d'onada"] = creixement(RES, exclude=None)[3:]
PROVES["Terme quadràtic comú"] = creixement(RES, extra=lambda t: pd.DataFrame({"cdd_quad": t["cdd0"] ** 2}))[3:]
if humitat is not None:
    td = humitat.set_index("data")["dew_point_2m_mean"]
    def ext_hum(t):
        x = t["data"].map(td).fillna(td.mean()).values - td.mean()
        return pd.DataFrame({"rosada": x, "cdd_x_rosada": t["cdd0"].values * x})
    PROVES["Punt de rosada (control + interacció)"] = creixement(RES, extra=ext_hum)[3:]
if preus is not None:
    pv = preus.set_index("data")["PVPC"]
    def ext_preu(t):
        x = (t["data"].map(pv) - pv.mean()) / pv.std()
        return pd.DataFrame({"preu": x.fillna(0).values, "cdd_x_preu": t["cdd0"].values * x.fillna(0).values})
    PROVES["Preu PVPC (control + interacció)"] = creixement(RES, extra=ext_preu)[3:]
tab_amb25, *_r25 = creixement(RES, years=ANYS)
PROVES["Amb el 2025 (fora de la regla)"] = tuple(_r25[2:])
PROVES["Només juny i setembre"] = creixement(RES, months=(6, 9))[3:]
PROVES["Només juliol i agost"] = creixement(RES, months=(7, 8))[3:]
ROB = pd.DataFrame(PROVES, index=["creixement", "se"]).T
ROB["z"] = ROB["creixement"] / ROB["se"]
print(f"\nCanvi de χ⁰ {A0}→{A1} (%/K) en cada prova:"); print(ROB.round(2).to_string())
CHI_2025 = tab_amb25.loc[2025, "chi0"] if 2025 in tab_amb25.index else np.nan
print(f"\nχ⁰ del 2025 (l'ÚNIC número del 2025 per als documents; fora de la regla): {CHI_2025:.2f} ± {tab_amb25.loc[2025, 'se']:.2f} %/K")

# Resposta absoluta (MWh/dia per K): residencial i total
ABS = {n: ct.absolute_response_by_year(NET[n], temp, TC, LAGS, ANYS_P, FESTIUS, exclude=ONADES) for n in ["Residencial", "Total"]}
print("\nResposta absoluta (MWh/dia per K):"); print(pd.DataFrame({n: a["resp"] for n, a in ABS.items()}).round(0).to_string())
tab_ser = creixement(NET["Serveis"])[0]
print("\nServeis (informatiu) · χ⁰ per estiu:", tab_ser["chi0"].round(2).to_dict())

cols = fs.year_colors(ANYS_P)
fig, axs = plt.subplots(1, 3, figsize=(16, 4.2), gridspec_kw={"width_ratios": [1, 0.8, 1.3]})
axs[0].errorbar(tab_res.index, tab_res["chi0"], yerr=2 * tab_res["se"], fmt="o-", color=fs.CAT[0], label="Residential")
tab_tot = creixement(NET["Total"])[0]
axs[0].errorbar(tab_tot.index + 0.08, tab_tot["chi0"], yerr=2 * tab_tot["se"], fmt="s--", color=fs.CAT[1], label="All sectors")
for a in FORA:
    if a in tab_amb25.index:
        axs[0].errorbar([a], [tab_amb25.loc[a, "chi0"]], yerr=[2 * tab_amb25.loc[a, "se"]], capsize=2.5,
                        label=f"{a} (fails the data-quality rule)", **fs.EXCLUDED)
axs[0].set_xticks(ANYS); axs[0].set_ylabel("χ⁰ (% per K)"); axs[0].set_title("a) Static response, each summer"); axs[0].legend(fontsize=8)
axs[1].hist(BOOT["Q"], bins=30, color=fs.CAT[0], alpha=0.8)
for v in IC["Q"]: axs[1].axvline(v, color=fs.INK_2, ls="--", lw=1)
axs[1].axvline(Q0, color=fs.INK, lw=1.5); axs[1].axvline(1, color=fs.CAT[1], lw=1)
axs[1].set_xlabel(f"Q = χ⁰{A1} / χ⁰{A0}"); axs[1].set_title("b) Block bootstrap (Tc re-chosen)")
y = np.arange(len(ROB))[::-1]
axs[2].errorbar(ROB["creixement"], y, xerr=2 * ROB["se"], fmt="o", color=fs.CAT[0])
EN = {"Base (residencial)": "Baseline (residential)", f"2022→{A1}": f"2022→{A1}", "Pendent (per any, ×3)": "Linear slope × 3 years",
      "Tots els sectors sumats": "All sectors summed", "Amb els dies d'onada": "Heatwave days included",
      "Terme quadràtic comú": "Common quadratic term", "Punt de rosada (control + interacció)": "Dew point (control + interaction)",
      "Preu PVPC (control + interacció)": "Electricity price (control + interaction)",
      "Amb el 2025 (fora de la regla)": "With 2025 (fails quality rule)", "Només juny i setembre": "June and September only",
      "Només juliol i agost": "July and August only"}
lab = [EN.get(n, n.replace("Estacions Meteocat", "Meteocat stations").replace("triat", "chosen")) for n in ROB.index]
axs[2].axvline(0, color=fs.INK_2, lw=0.8); axs[2].set_yticks(y); axs[2].set_yticklabels(lab, fontsize=8)
axs[2].set_xlabel(f"Change of χ⁰ {A0}→{A1} (% per K, ±2 s.e.)"); axs[2].set_title("c) Robustness (sign; magnitude in b)")
plt.tight_layout(); plt.savefig("figures_v14/fig2_growth_bootstrap_robustness.png"); plt.show()

## 9. R3 · Corbes sense forma imposada i model de llindars

**Què fa:**
1. Calcula, per a cada estiu, el consum en cada grau de temperatura (mitjana dels 3 últims dies) respecte dels dies ≤ 20 °C, **sense imposar cap forma** a la corba.
2. **Test pre-especificat (v0.9):** per sota de 24 °C, corbes iguals entre estius; per sobre, el **model de llindars**, és a dir, la mateixa forma × un factor λ per estiu (λ₂₀₂₁ = 1).
3. **Tota la corba (*post hoc*, 24/10/2026):** el model de llindars de 21 a 28 °C.

Tot amb els estius que passen la regla. També es dona amb el 2025, perquè a la v0.13 era el 2025 el que feia fallar la part de sota de 24 °C.

**Lectura física, com a hipòtesi:** si cada llar té un llindar d'encesa i la ciutat és la suma, la forma de la corba és la distribució de llindars i λ és la quantitat d'aparells per l'ús de cada aparell. Unes dades **compatibles amb** la mateixa forma i una λ més gran vol dir que no cal invocar un canvi del llindar de confort. **No ho demostra:** una forma igual és el que no podem rebutjar amb aquesta potència.

**Compte:** cada corba s'ancora en els 9-14 dies ≤ 20 °C del seu estiu. L'error de λ que dona l'ajust no inclou aquesta fragilitat; l'interval honest de λ és el del *bootstrap* (secció 8).

**Què cal mirar:** p > 0,05 vol dir que el model de llindars és compatible amb les dades.

In [ ]:
def proves_corbes(anys):
    Y, S = ct.response_curves(RES, temp, anys, FESTIUS, bins=range(21, 29), floor=20)
    baix, alt = [b for b in Y.index if b < 24], [b for b in Y.index if b >= 24]
    chi2_b, dof_b = ct.curves_equal(Y.loc[baix], S.loc[baix])
    lam_a, _, _, chi2_a, dof_a = ct.threshold_factor(Y.loc[alt], S.loc[alt], ref=A0)
    lam, se_lam, g, chi2_t, dof_t = ct.threshold_factor(Y, S, ref=A0)
    p = [1 - stats.chi2.cdf(c, d) for c, d in [(chi2_b, dof_b), (chi2_a, dof_a), (chi2_t, dof_t)]]
    return Y, S, lam, se_lam, p, chi2_t / dof_t

Y, S, lam, se_lam, (p_b, p_a, p_t), red_t = proves_corbes(ANYS_P)
print(f"Estius {ANYS_P}:")
print(f"  Pre-especificat: < 24 °C iguals p = {p_b:.2f} · ≥ 24 °C forma × factor p = {p_a:.2f}")
print(f"  Tota la corba (post hoc): p = {p_t:.2f}")
print("  λ (2021 = 1; error de l'ajust, sense la incertesa d'ancoratge):", {a: f"{lam[a]:.2f} ± {se_lam[a]:.2f}" for a in lam.index})
print(f"  λ{A1} amb interval de bootstrap: {lam[A1]:.2f} [{IC[f'lambda_{A1}'][0]:.2f} – {IC[f'lambda_{A1}'][1]:.2f}]")
Y5, S5, lam5, _, (p_b5, p_a5, p_t5), _ = proves_corbes(ANYS)
print(f"Amb el 2025: < 24 °C p = {p_b5:.2f} · ≥ 24 °C p = {p_a5:.2f} · tota la corba p = {p_t5:.2f} · λ2025 = {lam5[2025]:.2f}")

fig, axs = plt.subplots(1, 2, figsize=(12, 4.2))
for a in ANYS:
    st = dict(fmt="o-", color=cols[a]) if a in ANYS_P else dict(**fs.EXCLUDED)
    yy = (Y if a in ANYS_P else Y5)[a].dropna(); ee = (S if a in ANYS_P else S5)[a].dropna()
    axs[0].errorbar(np.r_[20, yy.index] + 0.06 * (a - 2023), np.r_[0, yy.values], yerr=np.r_[0, 2 * ee.values],
                    label=str(a) + ("" if a in ANYS_P else " (fails quality rule)"), **st)
axs[0].set_xlabel("Mean temperature of the last 3 days (°C; 20 = ≤20)"); axs[0].set_ylabel("Consumption above ≤20 °C (log × 100)")
axs[0].set_title("a) Response curve of each summer, no imposed shape"); axs[0].legend(fontsize=8)
axs[1].errorbar(lam.index, lam.values, yerr=2 * se_lam.values, fmt="o-", color=fs.CAT[0], label="fit s.e. (×2)")
axs[1].errorbar([A1 + 0.08], [lam[A1]], yerr=[[lam[A1] - IC[f"lambda_{A1}"][0]], [IC[f"lambda_{A1}"][1] - lam[A1]]],
                fmt="s", color=fs.CAT[1], label="bootstrap 95 %")
axs[1].set_xticks(ANYS_P); axs[1].set_ylabel("Amplitude λ (2021 = 1)"); axs[1].legend(fontsize=8)
axs[1].set_title(f"b) Amplitude of each summer (same shape, χ²/dof = {red_t:.2f})")
plt.tight_layout(); plt.savefig("figures_v14/fig3_curves_threshold_model.png"); plt.show()

## 10. R4 · Dia o nit: quina resposta creix (indici)

**Forçaments i retards:**
- **calor de nit** del dia $d$: T mitjana de **0 a 6 h del dia $d$**, és a dir, la nit que acaba el matí de $d$, per sobre de 20 °C. Retard 1: la nit anterior;
- **calor de dia** del dia $d$: T mitjana de **12 a 18 h del dia $d$**, per sobre de 27 °C. Retard 1: la tarda del dia anterior.

La resposta a cada calor és la suma dels dos retards. Per a la franja de consum de 0 a 6 h del dia $d$, la calor de dia del mateix dia és **posterior**: hi és com a control de la persistència, i l'efecte causal només pot venir de la tarda anterior.

**Què fa ara (revisió G1):**
1. **Test de la diferència** $D = \Delta\chi_\text{nit} - \Delta\chi_\text{dia}$ (2021→2024), amb la covariància completa de la regressió. Comptar quantes franges són significatives no és una prova.
2. **Autoconsum fotovoltaic:** Datadis mesura l'energia presa de la xarxa. Si les plaques creixen, retallen el consum de migdia dels dies assolellats. Afegim la radiació solar diària i la radiació × any com a controls.
3. **Tarifa 2.0TD** (des de l'1 de juny de 2021): els dies laborables, la vall és de 0 a 8 h; els caps de setmana i els festius nacionals, tot el dia és vall. Si el creixement nocturn fos un desplaçament de l'ús per preu, hauria de ser als laborables i no als caps de setmana.
4. **Graella de llindars** de dia (25-29 °C) i de nit (18-22 °C) per a la franja de 0 a 6 h.
5. **Alerta 2 i regla de qualitat:** dies del règim A de cada estiu (secció 5) i el mateix contrast 2021→2023 (només anys que passen la regla a les franges).

**Què cal mirar:** si $D$ té $z>2$ a la franja de 0 a 6 h i si es manté amb la radiació, als laborables i als caps de setmana, i a la graella. També si desapareix sense el 2024: en aquest cas, el resultat depèn de l'any amb el problema de franges.

In [ ]:
print("Dies útils del règim A per estiu (franges):", COB_F["dies"].astype(int).to_dict())
print("Fracció de CDD en dies del règim A:", COB_F["frac_cdd"].round(2).to_dict())
bandes = ct.band_temperatures(hor, {"nit": (0, 6), "dia": (12, 18)})
estiu_b = bandes[bandes["data"].dt.month.isin(SPEC["mesos_estiu"])]
print(f"Correlació T dia – T nit a l'estiu: {estiu_b[['t_dia', 't_nit']].corr().iloc[0, 1]:.2f}")
FORC = ct.day_night_forcings(bandes, SPEC["tc_dia"], SPEC["tc_nit"])
AF = [a for a in ANYS_P if a <= A1]                       # 2021-2024
A1F = max(a for a in ANYS_F if a <= A1)                   # últim any que passa la regla a les franges
RADX = ct.radiation_regressors(rad) if rad is not None else None

def fila(s, nom_franja, versio, years, y1, forc=None, **kw):
    bb, cc = ct.day_night_response(s, temp, FORC if forc is None else forc, TC, LAGS, years, FESTIUS, by_year=True, **kw)
    sm = ct.day_night_summary(bb, cc, True, A0, y1)
    D, sD, zD = ct.day_night_difference(bb, cc, A0, y1)
    return {"franja": nom_franja, "versió": versio, "fins": y1, "canvi_nit": sm["canvi_nit"], "se_nit": sm["se_canvi_nit"],
            "canvi_dia": sm["canvi_dia"], "se_dia": sm["se_canvi_dia"], "D": D, "se_D": sD, "z_D": zD}

files, nivells = [], []
for tr in FRANGES:
    s = ct.city_series(df, sector="Residencial", band=tr, days=DIES_BONS["Residencial"])
    b, c = ct.day_night_response(s, temp, FORC, TC, LAGS, AF, FESTIUS, exclude=REGIM_B)
    nivells.append({"franja": NOM_F[tr], **ct.day_night_summary(b, c)})
    files.append(fila(s, NOM_F[tr], "règim A (principal)", AF, A1, exclude=REGIM_B))
    files.append(fila(s, NOM_F[tr], f"règim A, fins al {A1F} (passa la regla)", [a for a in AF if a <= A1F], A1F, exclude=REGIM_B))
    files.append(fila(s, NOM_F[tr], "variable de règim", AF, A1, regime_b=REGIM_B))
    if RADX is not None:
        files.append(fila(s, NOM_F[tr], "règim A + radiació (FV)", AF, A1, exclude=REGIM_B, extra=RADX))
    files.append(fila(s, NOM_F[tr], "règim A, laborables", AF, A1, exclude=set(REGIM_B) | set(FESTIUS), weekdays=range(5)))
    cap = s[(s["data"].dt.dayofweek >= 5) | s["data"].isin(FESTIUS)]          # tot el dia és vall
    files.append(fila(cap, NOM_F[tr], "règim A, caps de setmana i festius", AF, A1, exclude=REGIM_B))
NIV = pd.DataFrame(nivells).set_index("franja")
DN = pd.DataFrame(files).set_index(["franja", "versió"])
print("\nResposta (%/K) a la calor de dia i de nit, règim A, 2021-2024:"); print(NIV.round(2).to_string())
print(f"\nCanvi {A0}→fins (%/K) i test de la diferència D = Δnit − Δdia:"); print(DN.round(2).to_string())

# Graella de llindars (franja 0-6 h, règim A)
s0 = ct.city_series(df, sector="Residencial", band=FRANGES[0], days=DIES_BONS["Residencial"])
GR = []
for tcd in SPEC["graella_tc_dia"]:
    for tcn in SPEC["graella_tc_nit"]:
        r = fila(s0, NOM_F[FRANGES[0]], "graella", AF, A1, exclude=REGIM_B, forc=ct.day_night_forcings(bandes, tcd, tcn))
        GR.append({"tc_dia": tcd, "tc_nit": tcn, "D": r["D"], "z_D": r["z_D"]})
GR = pd.DataFrame(GR)
print("\nGraella (0-6 h): z de D"); print(GR.pivot(index="tc_dia", columns="tc_nit", values="z_D").round(1).to_string())
print(f"z > 2 en {int((GR['z_D'] > 2).sum())} de {len(GR)} combinacions")

princ = DN.xs("règim A (principal)", level="versió")
x = np.arange(len(princ))
fig, axs = plt.subplots(1, 3, figsize=(17, 4.2), gridspec_kw={"width_ratios": [1, 1.2, 0.9]})
for i, (nom, lab) in enumerate([("dia", "Day heat (12-18 h)"), ("nit", "Night heat (0-6 h)")]):
    axs[0].bar(x + 0.38 * i, NIV[nom], 0.36, yerr=2 * NIV[f"se_{nom}"], color=fs.CAT[i], label=lab, error_kw={"ecolor": fs.INK_2, "elinewidth": 1})
    axs[1].bar(x + 0.27 * i, princ[f"canvi_{nom}"], 0.25, yerr=2 * princ[f"se_{nom}"], color=fs.CAT[i], label=f"Change: {lab.lower()}", error_kw={"ecolor": fs.INK_2, "elinewidth": 1})
axs[1].bar(x + 0.54, princ["D"], 0.25, yerr=2 * princ["se_D"], color=fs.INK_2, label="Difference night − day", error_kw={"ecolor": fs.INK, "elinewidth": 1})
for ax, t, off in [(axs[0], "a) Response of each time band, 2021-2024", 0.19), (axs[1], f"b) Change {A0}→{A1} and its difference (±2 s.e.)", 0.27)]:
    ax.set_xticks(x + off); ax.set_xticklabels(princ.index); ax.axhline(0, color=fs.INK_2, lw=0.8)
    ax.set_xlabel("Consumption time band (h)"); ax.set_ylabel("% per K"); ax.set_title(t); ax.legend(fontsize=8)
pv_ = GR.pivot(index="tc_dia", columns="tc_nit", values="z_D")
im = axs[2].imshow(pv_.values, cmap="Blues", vmin=0, vmax=4, origin="lower")
for (i, j), v in np.ndenumerate(pv_.values): axs[2].text(j, i, f"{v:.1f}", ha="center", va="center", fontsize=8, color=fs.INK if v < 2.5 else "white")
axs[2].set_xticks(range(len(pv_.columns))); axs[2].set_xticklabels([f"{c:.0f}" for c in pv_.columns])
axs[2].set_yticks(range(len(pv_.index))); axs[2].set_yticklabels([f"{c:.0f}" for c in pv_.index]); axs[2].grid(False)
axs[2].set_xlabel("Night threshold (°C)"); axs[2].set_ylabel("Day threshold (°C)"); axs[2].set_title("c) z of the difference, 0-6 h band")
plt.tight_layout(); plt.savefig("figures_v14/fig4_day_night.png"); plt.show()

## 11. R5 · El fred i la simetria de la V

**Què fa:** el model d'hivern amb graus-dia de calefacció $\mathrm{HDD}=\max(T_h-T,0)$ i efectes fixos d'hivern, setmana ISO (absorbeix el cicle de la llum), dia, festiu i Nadal. Tria $T_h$ igual que $T_c$. Estima $\chi_h$ per a cada hivern amb la **finestra pre-especificada (novembre-març)** i calcula $R=\chi_h/\chi_c$.

**Annex (*post hoc*):** desembre-febrer i novembre-març amb efectes fixos d'hivern × mes. Només serveixen per dir que el **canvi** de $\chi_h$ no és robust. No surten a la figura del cos.

**Compte:** el gas no el veiem. $\chi_h$ només mesura la part elèctrica de la calefacció.

**Què cal mirar:** el nivell de $\chi_h$, que és el resultat robust. Amb la v0.14 (estius 2021-2024 i finestra pre-especificada), $R$ **no baixa** (0,60 → 0,57). Només les finestres *post hoc* la fan baixar. Per tant, «la V es torna menys simètrica» surt de l'article (**decisió de Pau, 25/10/2026**), i el canvi de $\chi_h$ tampoc hi és.

In [ ]:
TH, perfil_h = ct.choose_th(RES, temp, SPEC["th_candidats"], LAGS, FESTIUS, SPEC["hiverns"])
print(f"Th = {TH} °C (l'error varia un {100 * (perfil_h['mse'].max() / perfil_h['mse'].min() - 1):.0f} % entre llindars)")
H = {}
for nom, kw in [("nov.-març (pre-especificat)", dict(months=SPEC["mesos_hivern"])),
                ("annex: des.-feb. (post hoc)", dict(months=SPEC["mesos_hivern_cor"])),
                ("annex: nov.-març + hivern × mes (post hoc)", dict(months=SPEC["mesos_hivern"], winter_month=True))]:
    tab, b, c = ct.response_by_winter(RES, temp, TH, LAGS, SPEC["hiverns"], FESTIUS, **kw)
    d, se = ct.growth(b, c, SPEC["hiverns"][0], SPEC["hiverns"][-1], LAGS, prefix="hdd")
    H[nom] = tab.assign(canvi=d, se_canvi=se)
    print(f"  {nom:42s} χ_h: " + " / ".join(f"{v:.2f}" for v in tab["chi_h"]) + f"  · canvi {d:+.2f} ± {se:.2f}")
pool = ct.fit_winter(RES, temp, TH, LAGS, FESTIUS, SPEC["hiverns"])
print(f"Nivell (tots els hiverns, nov.-març): χ_h = {pool['chi_h']:.2f} ± {pool['se']:.2f} %/K")

SIM = []
for nom, tab in H.items():
    for w in SPEC["hiverns"]:
        if w in tab_res.index:
            Rv, sR = ct.asymmetry(tab.loc[w, "chi_h"], tab.loc[w, "se"], tab_res.loc[w, "chi0"], tab_res.loc[w, "se"])
            SIM.append({"versió": nom, "any": w, "R": Rv, "se": sR})
SIM = pd.DataFrame(SIM)
print("\nR = χ_h (hivern que comença el nov. de l'any) / χ_c (estiu de l'any):")
print(SIM.pivot(index="any", columns="versió", values="R").round(2).to_string())

PRE = "nov.-març (pre-especificat)"
fig, axs = plt.subplots(1, 2, figsize=(12, 4))
axs[0].errorbar(H[PRE].index, H[PRE]["chi_h"], yerr=2 * H[PRE]["se"], fmt="o-", color=fs.CAT[0])
g_ = SIM[SIM["versió"] == PRE]
axs[1].errorbar(g_["any"], g_["R"], yerr=2 * g_["se"], fmt="o-", color=fs.CAT[0])
axs[0].set_ylabel(r"$\chi_h$ (% per K below $T_h$)"); axs[0].set_title("a) Response to cold, Nov-Mar (winter starting year)")
axs[1].set_ylabel(r"$R = \chi_h / \chi_c$"); axs[1].set_title("b) Symmetry of the V")
for ax in axs: ax.set_xticks(SPEC["hiverns"])
plt.tight_layout(); plt.savefig("figures_v14/fig5_cold_symmetry.png"); plt.show()

## 12. R6 · Histèresi: test de fronts

**Què fa:** comprova si, a la mateixa temperatura, el consum és més alt quan la calor se'n va que quan arriba, un cop descomptada la memòria. El terme $\gamma\,\mathrm{CDD}(d)\,\mathbf 1[T(d)<T(d-1)]$ s'afegeix a una memòria additiva flexible (frontisses per retard). Una memòria lineal o corbada dona $\gamma = 0$; la histèresi de relés (aire encès a 25 °C i apagat a 23 °C) dona $\gamma > 0$.

**Calibratge amb dades sintètiques:** falsos positius del 0-3 %; potència del 100 % amb una histèresi d'1 K, del 62 % amb 2 K i del 10 % amb 3 K. **Un nul vol dir «no hi ha histèresi estreta».**

**Què cal mirar:** $|z| < 2$ a totes les versions.

In [ ]:
HIS = {}
for nom, kw in [("base", {}), ("sense onades", dict(exclude=ONADES)), (f"Tc = {TC - 1}", dict(tc=TC - 1)),
                (f"Tc = {TC + 1}", dict(tc=TC + 1)), ("estius 2021-22", dict(years=[2021, 2022])),
                ("estius 2023-24", dict(years=[2023, 2024]))]:
    kw = dict(kw); tc_ = kw.pop("tc", TC); yrs = kw.pop("years", ANYS_P)
    h = ct.hysteresis_test(RES, temp, tc_, LAGS, FESTIUS, SPEC["mesos_estiu"], yrs, **kw)
    HIS[nom] = {k: h[k] for k in ["gamma", "se", "z", "n"]}
HIS = pd.DataFrame(HIS).T
print("γ (% per K de més els dies que la temperatura baixa):"); print(HIS.round(3).to_string())

## 13. Component elèctric de la calor antropogènica

**Què fa:** converteix la resposta absoluta de **tots els sectors** (MWh/dia per K, secció 8) en un flux per unitat d'àrea:
$$\frac{\partial Q_{F,\mathrm{el}}}{\partial T} = \frac{\Delta E\ [\mathrm{MWh/dia/K}] \times 10^6}{24\ \mathrm{h} \times A}\quad [\mathrm{W\,m^{-2}\,K^{-1}}]$$
amb $A$ = 101 km² (l'àrea dels 42 CP, una reconstrucció).

**És només el component elèctric de $Q_F$:** no hi ha gas, trànsit ni metabolisme. Suposa que tota l'electricitat consumida acaba com a calor a la ciutat el mateix dia. Llavors $G = (\partial Q_{F,\mathrm{el}}/\partial T)/h_\text{eff}$.

**Compte:** $h_\text{eff}$ (10-50 W m⁻² K⁻¹) **no està verificat**: $G$ és només un ordre de magnitud per a la discussió, no un resultat.

In [ ]:
a_m2 = SPEC["area_km2"] * 1e6
QF = (ABS["Total"]["resp"] * 1e6 / 24 / a_m2).rename("dQFel_dT_W_m2_K").to_frame()
QF["se"] = ABS["Total"]["se"] * 1e6 / 24 / a_m2
QF["G_min"], QF["G_max"] = QF.iloc[:, 0] / SPEC["h_eff"][1], QF.iloc[:, 0] / SPEC["h_eff"][0]
print("∂Q_F,el/∂T (W m⁻² K⁻¹), component elèctric, i G per a h_eff = 50 i 10 W m⁻² K⁻¹ (NO verificat):")
print(QF.round(3).to_string())

## 14. Predicció registrada de l'estiu 2026

**Per què:** amb quatre estius no es pot distingir una tendència d'un salt. La millor prova possible és una **predicció fixada abans de tenir les dades** de l'estiu 2026. Els valors són a `docs/prediccio_2026.md` del repositori, en un *commit* amb data del 24/10/2026, i aquesta cel·la els copia.

**Dues hipòtesis, amb la mateixa canonada (paquet v0.4.0, $T_c$ fix):**
- **H1, continuació lineal:** $\chi^0_{2026}$ és l'extrapolació de la recta dels estius 2021-2024. L'interval de predicció fa servir la dispersió entre anys.
- **H0, salt i replà:** $\chi^0_{2026}$ es queda al nivell del 2024.

**Condicions, fixades ara:** l'estiu 2026 ha de passar la regla de qualitat (secció 5). Es fa servir el mateix $T_c$, sense tornar-lo a triar, i les mateixes exclusions.

**Avís honest:** els dos intervals se superposen. Un resultat a la zona comuna no discrimina; només un valor fora d'un dels dos intervals és informatiu.

**Quan hi hagi dades:** posa el CSV del 2026 a `dades_tesi` (nom amb `consum_electricitat` i `2026`) i torna a executar el notebook. La cel·la fa la comparació sola.

In [ ]:
# Valors registrats el 24/10/2026 (copiats de docs/prediccio_2026.md; NO es recalculen)
PRED_2026 = {"tc": 20.0, "H1_chi0": 9.09, "H1_pi95": (6.79, 11.38), "H0_chi0": 7.48, "H0_ic95": (6.44, 8.21),
             "H1_lambda": 2.26, "data_registre": "2026-10-24"}

# Com es va calcular (es repeteix per comprovar que la canonada dona el mateix)
x = np.array(ANYS_P) - A0; yv = tab_res.loc[ANYS_P, "chi0"].values
co = np.polyfit(x, yv, 1); res_ = yv - np.polyval(co, x); s_ = np.sqrt(np.sum(res_ ** 2) / (len(x) - 2))
X_ = np.vstack([np.ones_like(x), x]).T; v_ = np.array([1, 2026 - A0])
pi = stats.t.ppf(0.975, len(x) - 2) * s_ * np.sqrt(1 + v_ @ np.linalg.inv(X_.T @ X_) @ v_)
lam_lin = np.polyval(np.polyfit(x, lam.loc[ANYS_P].values, 1), 2026 - A0)
print(f"Recalculat ara: H1 χ⁰₂₀₂₆ = {np.polyval(co, 2026 - A0):.2f} [{np.polyval(co, 2026 - A0) - pi:.2f} – {np.polyval(co, 2026 - A0) + pi:.2f}] · "
      f"H0 = {tab_res.loc[A1, 'chi0']:.2f} [{IC[f'chi0_{A1}'][0]:.2f} – {IC[f'chi0_{A1}'][1]:.2f}] · λ₂₀₂₆ lineal = {lam_lin:.2f}")
print("Registrat:", PRED_2026)
if TC != PRED_2026["tc"]:
    print(f"[!] El Tc d'aquesta execució ({TC}) no és el registrat: la comparació es fa amb el registrat")

f26 = [f for f in Path(CARPETA).glob("*.csv") if "consum_electricitat" in f.name.lower() and "2026" in f.name]
if f26:
    d26 = pd.concat([ct.daily_consumption(pd.read_csv(f)) for f in f26])
    fi26 = d26["data"].max().strftime("%Y-%m-%d")
    t26 = ct.from_cache(CARPETA, "temperatura_2026.csv", lambda: ct.era5_daily(lat, lon, "2026-01-01", fi26), until=fi26)
    tempx = pd.concat([temp[["data", "tmax", "tmean"]], t26[["data", "tmax", "tmean"]]]).drop_duplicates("data").sort_values("data").reset_index(drop=True)
    # onades del 2026 amb el MATEIX llindar P95 dels estius 2021-2025 (no es recalcula)
    p95 = temp[temp["data"].dt.month.isin(SPEC["mesos_estiu"])]["tmax"].quantile(SPEC["onada_percentil"])
    calent = tempx["tmax"] >= p95; ratxa = calent.groupby((calent != calent.shift()).cumsum()).transform("sum")
    ONADES_X = ONADES | set(tempx.loc[calent & (ratxa >= SPEC["onada_dies"]) & (tempx["data"].dt.year == 2026), "data"])
    FEST_X = sorted(set(FESTIUS) | set(pd.to_datetime(list(holidays.country_holidays("ES", subdiv="CT", years=[2026]).keys()))))
    s26 = pd.concat([SERIES["Residencial"], ct.city_series(d26, sector="Residencial")]).drop_duplicates("data")
    bons26, _ = ct.clean_days(s26, tempx, FEST_X, **kw_an)
    cob26 = ct.summer_coverage(bons26, tempx, 20.0, [2026])
    print("Cobertura del 2026:", cob26.round(3).to_dict("index"))
    if ct.quality_rule(cob26, SPEC["regla_min_cdd"]):
        tb26, _, _ = ct.response_by_year(s26[s26["data"].isin(bons26)], tempx, PRED_2026["tc"], LAGS, ANYS_P + [2026], FEST_X, exclude=ONADES_X)
        v26 = tb26.loc[2026, "chi0"]
        print(f">>> χ⁰₂₀₂₆ observat = {v26:.2f} ± {tb26.loc[2026, 'se']:.2f} %/K · "
              f"dins de H1: {PRED_2026['H1_pi95'][0] <= v26 <= PRED_2026['H1_pi95'][1]} · dins de H0: {PRED_2026['H0_ic95'][0] <= v26 <= PRED_2026['H0_ic95'][1]}")
    else:
        print("[!] El 2026 no passa la regla de qualitat: la predicció no es pot avaluar")
else:
    print("Encara no hi ha dades del 2026: la predicció queda pendent.")

## 15. Resum de resultats, figures i exportació

**Què fa:** reuneix tots els números de l'article en una sola taula, desa les figures (en anglès, 300 ppp) i guarda un `.zip` al Drive perquè Claude pugui revisar-ho tot.

In [ ]:
nit = DN.loc[("00-06", "règim A (principal)")]
nit_niv = NIV.loc["00-06"]
fv = DN.loc[("00-06", "règim A + radiació (FV)")] if ("00-06", "règim A + radiació (FV)") in DN.index else None
lab_, cap_ = DN.loc[("00-06", "règim A, laborables")], DN.loc[("00-06", "règim A, caps de setmana i festius")]
fins_ = DN.loc[("00-06", f"règim A, fins al {A1F} (passa la regla)")]
RESUM = pd.DataFrame([
    ["Q", "Regla de qualitat: estius que entren", f"{ANYS_P}", f"fora: {FORA}; franges: {ANYS_F}"],
    ["R1", "Tc (°C)", f"{TC}", f"bootstrap: {IC['tc'][0]:.1f}–{IC['tc'][1]:.1f}"],
    ["R1", "χ⁰ tots els estius (%/K)", f"{r1['chi0']:.2f} [{IC['chi0_tots'][0]:.2f}–{IC['chi0_tots'][1]:.2f}]", "IC bootstrap amb Tc a dins"],
    ["R1", "Residu dels dies d'onada (fora de l'ajust) (%)", f"{ones['residu_pct'].mean():+.1f}", f"{len(ones)} dies; dispersió normal ±{sd_ones:.1f} %"],
    ["R2", f"MÈTRICA PRINCIPAL Q = χ⁰{A1}/χ⁰{A0}", f"{Q0:.2f} [{IC['Q'][0]:.2f}–{IC['Q'][1]:.2f}]", "bootstrap per setmanes, Tc a dins"],
    ["R2", f"χ⁰ {A0} → {A1} (%/K)", f"{tab_res.loc[A0, 'chi0']:.2f} → {tab_res.loc[A1, 'chi0']:.2f}", f"diferència {d0:+.2f} [{IC['dif'][0]:+.2f}–{IC['dif'][1]:+.2f}]"],
    ["R2", "χ⁰ 2025 (fora de la regla)", f"{CHI_2025:.2f}", "l'únic número del 2025"],
    ["R2", "Canvi, tots els sectors (%/K)", "{:+.2f} ± {:.2f}".format(*PROVES["Tots els sectors sumats"]), "prova de composició"],
    ["R2", "Proves de robustesa amb z > 2", f"{int((ROB['z'] > 2).sum())} de {len(ROB)}", "juny + setembre: no s'espera"],
    ["R2", f"Resposta absoluta total {A0} → {A1} (MWh/dia/K)", f"{ABS['Total'].loc[A0, 'resp']:.0f} → {ABS['Total'].loc[A1, 'resp']:.0f}", ""],
    ["R3", "Pre-especificat: < 24 °C iguals (p) / forma × factor ≥ 24 °C (p)", f"{p_b:.2f} / {p_a:.2f}", f"amb 2025: {p_b5:.2f} / {p_a5:.2f}"],
    ["R3", "Forma × factor a tota la corba 21-28 °C (p)", f"{p_t:.2f}", f"post hoc · λ{A1} = {lam[A1]:.2f} [{IC[f'lambda_{A1}'][0]:.2f}–{IC[f'lambda_{A1}'][1]:.2f}]"],
    ["R4", "0-6 h: resposta a la calor de nit / de dia (%/K)", f"{nit_niv['nit']:.2f} / {nit_niv['dia']:.2f}", "règim A, 2021-2024"],
    ["R4", f"0-6 h: D = Δnit − Δdia {A0}→{A1} (%/K)", f"{nit['D']:+.2f} ± {nit['se_D']:.2f} (z = {nit['z_D']:.1f})", "covariància completa"],
    ["R4", f"0-6 h: D fins al {A1F} (anys que passen la regla)", f"{fins_['D']:+.2f} ± {fins_['se_D']:.2f} (z = {fins_['z_D']:.1f})", "sense el 2024"],
    ["R4", "0-6 h: D amb radiació (FV)", "—" if fv is None else f"{fv['D']:+.2f} ± {fv['se_D']:.2f} (z = {fv['z_D']:.1f})", ""],
    ["R4", "0-6 h: D laborables / caps de setmana", f"{lab_['D']:+.2f} (z = {lab_['z_D']:.1f}) / {cap_['D']:+.2f} (z = {cap_['z_D']:.1f})", "tarifa 2.0TD"],
    ["R4", "Graella de llindars: combinacions amb z > 2", f"{int((GR['z_D'] > 2).sum())} de {len(GR)}", ""],
    ["R4", "Franges amb D z > 2", f"{int((princ['z_D'] > 2).sum())} de {len(princ)}", "règim A"],
    ["R5", "χ_h nivell (%/K), Th", f"{pool['chi_h']:.2f} ± {pool['se']:.2f}, {TH} °C", "nov.-març"],
    ["R5", "Canvi de χ_h (pre-especificat; annex)", " / ".join(f"{t['canvi'].iloc[0]:+.2f}" for t in H.values()), "no robust"],
    ["R6", "γ histèresi (base)", f"{HIS.loc['base', 'gamma']:+.3f} ± {HIS.loc['base', 'se']:.3f}", f"màx |z| = {HIS['z'].abs().max():.1f}"],
    ["—", f"∂Q_F,el/∂T total {A1} (W m⁻² K⁻¹)", f"{QF.iloc[:, 0].loc[A1]:.2f}", "només elèctric; G no verificat"],
    ["P", "Predicció 2026 registrada: H1 / H0 (χ⁰, %/K)", f"{PRED_2026['H1_chi0']:.2f} / {PRED_2026['H0_chi0']:.2f}", "docs/prediccio_2026.md"],
], columns=["resultat", "magnitud", "valor", "nota"])
print(RESUM.to_string(index=False))
RESUM.to_csv("resum_article1_v14.csv", index=False)

# Exportació (també al Drive, perquè la descàrrega de Colab de vegades no surt)
SORT = "dades_per_claude_v14"; os.makedirs(SORT, exist_ok=True)
for nom, s in SERIES.items(): s.to_csv(f"{SORT}/serie_{nom.lower()}.csv", index=False)
pd.Series(sorted(set().union(*[set(SERIES[n]['data']) - DIES_BONS[n] for n in SERIES])), name="data").to_csv(f"{SORT}/dies_exclosos.csv", index=False)
df.groupby(["data", "sector", "tram"], as_index=False)["kwh"].sum().to_csv(f"{SORT}/ciutat_sector_franja.csv.gz", index=False, compression="gzip")
temp.to_csv(f"{SORT}/temp_era5.csv", index=False); bandes.to_csv(f"{SORT}/temp_franges.csv", index=False)
if T_EST is not None: T_EST.to_csv(f"{SORT}/temp_estacions.csv", index=False)
if rad is not None: rad.to_csv(f"{SORT}/radiacio.csv", index=False)
pd.Series(sorted(REGIM_B), name="data").to_csv(f"{SORT}/regim_b.csv", index=False)
for f in ["resum_article1_v14.csv", "registre_alertes_v14.csv", "dies_exclosos_v14.csv", "regla_qualitat_v14.csv"]: shutil.copy(f, SORT)
TAULES = {"r1_funcio_resposta": pd.DataFrame({"chi_k": chi_k, "se": se_k, "chi_k_correlacions": cor}),
          "r1_onades": ones, "r2_chi_per_estiu": tab_res, "r2_chi_per_estiu_amb_2025": tab_amb25, "r2_robustesa": ROB,
          "r2_bootstrap": BOOT, "r2_resposta_absoluta": pd.concat(ABS, axis=1), "r3_corbes": Y, "r3_errors_corbes": S,
          "r3_lambda": pd.DataFrame({"lambda": lam, "se": se_lam}), "r4_nivells": NIV, "r4_dia_nit": DN, "r4_graella": GR,
          "r5_fred_per_hivern": pd.concat(H), "r5_simetria": SIM, "r6_histeresi": HIS, "qf_sensibilitat": QF}
for nom, t in TAULES.items():
    t.to_csv(f"{SORT}/{nom}.csv")
json.dump({k: (list(v) if isinstance(v, tuple) else v) for k, v in SPEC.items()} | {"TC": TC, "TH": TH, "ANYS_P": ANYS_P, "PRED_2026": PRED_2026},
          open(f"{SORT}/especificacio.json", "w"), ensure_ascii=False, default=float)
with zipfile.ZipFile("dades_per_claude_v14.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for f in os.listdir(SORT): z.write(f"{SORT}/{f}", f)
    for f in os.listdir("figures_v14"): z.write(f"figures_v14/{f}", f"figures/{f}")
try:
    shutil.copy("dades_per_claude_v14.zip", f"{CARPETA}/dades_per_claude_v14.zip")
    print(f"\n[OK] dades_per_claude_v14.zip ({os.path.getsize('dades_per_claude_v14.zip') / 1e6:.1f} MB) desat a {CARPETA}: adjunta'l al xat")
except Exception as e:
    print(f"[!] No s'ha pogut desar al Drive ({e}): baixa'l del panell de fitxers de Colab")